# 21c - Health check Part C: the final field test, scored once

Every field number so far comes from the selection set, which also fitted the calibration
and was used to pick the threshold, so those numbers may be optimistic. This notebook scores
the **deployed system, exactly as it runs**, on field photos that played **no part in any
choice**, and reports the result. It is the last evaluation before the report.

| Source | Photos used | Licence |
|---|---|---|
| Tomato leaf diseases, Bangladesh, iPhone 11 in the field (Mendeley 93h9p62kg4) | bacterial spot, early blight, late blight, healthy | CC BY 4.0 |
| Crop pest and disease dataset, farms in Ghana (Mendeley bwh3zbpkpv) | healthy tomato, healthy maize (false alarms) | CC BY 4.0 |

Details and the sources checked and not used are in `src/data/final_test.py`.

**Rules, written on 31 Oct 2026 before any photo was downloaded:**

- Scored **once**. Nothing deployed (model, calibration, threshold, leaf or blur check, photo
  size) is changed because of these results; a surprising result is investigated and
  reported, not tuned away. The notebook refuses to score a second time unless a reason is
  written down.
- Labels come only from the folder names (mapping fixed in `src/data/final_test.py`), and
  you look at sample sheets of every class before anything is scored.
- Duplicates of any photo used so far (training sources, PlantDoc, selection set, D17 and
  D22 photos), also turned or mirrored, are removed; near-copies within the new photos are
  reduced to one.
- The deployed pipeline: the phone's photo (the setting kept or chosen in notebook 21b), the
  server's JPEG, the leaf check (cut-off 0.9975), the blur check if notebook 21b switched it
  on, then the ensemble with the within-crop confidence (temperature 1.7002, threshold
  0.9557, crop share at least 0.5). The farmer's crop is given.

**Run notebook 21b first.** A GPU runtime is best (T4); about 30-60 minutes, most of it the
download. Nothing large is saved to Drive: the photos stay on Colab's disk.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## The deployed settings, and the once-only guard

In [ ]:
import importlib
import io
import json
import re
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

from src import blur_check, calibration as cal, leaf_check, metrics, training
from src.data import cleaning, d17, final_test, selection_set
for module in (blur_check, cal, leaf_check, metrics, training, cleaning, selection_set, d17, final_test):
    importlib.reload(module)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BOOTSTRAP, SEED = 1000, config.RANDOM_SEED
RESULTS = ML_DIR / "results"
LOCAL = Path("/content/final_test")
OUT_DIR = config.MODELS_DIR / "final_field_test"
REPO_RESULTS = RESULTS / "final_field_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Scored once: a second run must say why (e.g. "Colab disconnected before the results were saved").
RERUN_REASON = ""
if (REPO_RESULTS / "decision.json").exists() or (OUT_DIR / "decision.json").exists():
    assert RERUN_REASON.strip(), "This test was already scored once. Write RERUN_REASON to run it again."

health = json.loads((RESULTS / "health_check" / "decision.json").read_text())
photos_b = RESULTS / "health_check_photos"
assert (photos_b / "decision.json").exists(), "Run notebook 21b (and merge its results) first."
part_b = json.loads((photos_b / "decision.json").read_text())
blur = json.loads((photos_b / "blur_check.json").read_text())
leaf = json.loads((RESULTS / "leaf_check" / "leaf_check.json").read_text())

MEMBERS = health["deployed"]["model_runs"]
TEMPERATURE = health["deployed"]["temperature"]
CROP_TEMPERATURE = health["A3"]["temperature_within_crop"]
THRESHOLD = health["A3"]["threshold"]
APP_SIDE, APP_QUALITY = map(int, re.match(r"(\d+) px, q(\d+)", part_b["B1"]["app_setting"]).groups())
SERVER_QUALITY = 85
BLUR_ON, BLUR_CUT_OFF = bool(blur["switched_on"]), float(blur["cut_off"])
LEAF_CUT_OFF = float(leaf["threshold"])

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ").strip() for c in CLASSES}
print(f"Photo {APP_SIDE} px q{APP_QUALITY} -> server q{SERVER_QUALITY} | leaf cut-off {LEAF_CUT_OFF} | "
      f"blur check {'ON, ' + str(round(BLUR_CUT_OFF, 1)) if BLUR_ON else 'off'} | T {TEMPERATURE:.4f} | "
      f"crop T {CROP_TEMPERATURE:.4f} | threshold {THRESHOLD:.4f} | {DEVICE}")

## Licence, download and the photos used

Files go to Colab's disk only and are checked against Mendeley's SHA-256. Files named as
augmented are not downloaded. **Check the folder table**: every folder is listed with what
it became (a class, or left out and why).

In [ ]:
records, tables = [], []
for name, source in final_test.SOURCES.items():
    licence = final_test.check_licence(source)
    for record in final_test.download(name, LOCAL / "download" / name):
        records.append({**record, "licence": licence, "citation": source.citation})
        if "sha256" in record:
            final_test.extract(name, LOCAL / "download" / name / record["file"], LOCAL / "unpacked" / name)
    tables.append(final_test.inventory(name, LOCAL / "unpacked" / name))
sources_table = pd.DataFrame(records)
display(sources_table.drop(columns=["citation"]))
found = pd.concat(tables, ignore_index=True)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 300)
display(final_test.folder_summary(found))

## Duplicate checks

Against every photo used so far: the training sources and PlantDoc (`clean_manifest.csv`),
the selection set (`selection_manifest.csv`) and the photos D17 and D22 added to training.

In [ ]:
def kept_rows(file):
    table = pd.read_csv(config.METADATA_DIR / file, usecols=["md5", "phash", "keep"])
    return table[table["keep"].astype(str) == "True"][["md5", "phash"]]


existing = pd.concat([
    pd.read_csv(config.METADATA_DIR / "clean_manifest.csv", usecols=["md5", "phash"]),
    pd.read_csv(config.METADATA_DIR / "selection_manifest.csv", usecols=["md5", "phash"]),
    kept_rows("d17_manifest.csv"), kept_rows("d22_manifest.csv"),
], ignore_index=True)
print(f"Compared against {len(existing)} photos used so far")

mapped = found[found["class"].notna()].reset_index(drop=True)
prints = selection_set.fingerprint_all(list(mapped["path"]))
checked = pd.concat([mapped, prints], axis=1)
unmapped = found[found["class"].isna()].assign(error=None, md5=None, phash=None, phash_variants=None,
                                               width=None, height=None)
result = selection_set.deduplicate(pd.concat([checked, unmapped], ignore_index=True), existing,
                                   config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)
result = d17.drop_copies_of_matches(result, config.NEAR_DUPLICATE_MAX_DISTANCE)
result["reason"] = result["reason"].replace("near_duplicate_within_selection", "near_duplicate_within_new_photos")
reasons = (result.assign(reason=result["reason"].replace("", "kept"))
           .groupby(["source", "reason"]).size().rename("photos").reset_index())
display(reasons)
kept = result[result["keep"]].reset_index(drop=True)
kept["crop"] = CLASS_CROPS[kept["class"].map(INDEX).to_numpy()]
display(kept.groupby(["source", "class"]).size().rename("kept").to_frame())

## STOP: look at the photos

One sheet per source and class, 18 random kept photos each. Check that each sheet shows
**real field photos of that crop's leaves** with that condition:

- **tomato bacterial spot:** many small, dark, water-soaked spots, often with a yellow edge;
- **tomato early blight:** brown spots with rings like a target, often on older leaves;
- **tomato late blight:** large, dark, greasy-looking patches, sometimes with white mould;
- **healthy (tomato or maize):** a green leaf without spots.

If a sheet looks wrong (another crop, another disease, lab photos on plain backgrounds,
drawings or screenshots), stop and describe it to Claude. Nothing has been scored yet.

In [ ]:
REVIEW_DIR = LOCAL / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sheets = []
for (name, cls), paths in d17.review_sample(kept, 18, SEED).items():
    sheet = d17.contact_sheet(paths, f"{name} | {SHORT[cls]} | {int(((kept['source'] == name) & (kept['class'] == cls)).sum())} kept")
    target = REVIEW_DIR / f"{name}__{SHORT[cls].replace(' ', '_')}.jpg"
    sheet.save(target, quality=85)
    sheets.append(target)
    display(sheet)

In [ ]:
# After looking at EVERY sheet above, change False to True and run this cell.
I_CHECKED_THE_PHOTOS = False

assert I_CHECKED_THE_PHOTOS, "Look at every sheet above first, then set I_CHECKED_THE_PHOTOS = True."
print("Checked. Scoring next.")

## Score once, exactly as the deployed system does

Each photo is made into what the server would store (the phone's size and JPEG, then the
server's quality 85), saved on Colab's disk, and then passed through the leaf check, the
blur check if it is on, and the ensemble.

In [ ]:
def jpeg(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    with Image.open(buffer) as again:
        return again.convert("RGB")


STORED = LOCAL / "stored"
STORED.mkdir(parents=True, exist_ok=True)
stored_paths, sharpness = [], []
for i, path in enumerate(tqdm(kept["path"], desc="Phone and server")):
    with Image.open(path) as original:
        image = original.convert("RGB")
    image.thumbnail((APP_SIDE, APP_SIDE), Image.BICUBIC)
    image = jpeg(jpeg(image, APP_QUALITY), SERVER_QUALITY)
    target = STORED / f"{i:05d}.jpg"
    image.save(target, format="JPEG", quality=100)
    stored_paths.append(target)
    sharpness.append(blur_check.sharpness(image))
sharpness = np.array(sharpness)

head = {"mean": np.array(leaf["feature_mean"]), "scale": np.array(leaf["feature_scale"]),
        "coef": np.array(leaf["coef"]), "intercept": leaf["intercept"]}
features, readable = leaf_check.features(leaf_check.backbone(leaf["architecture"]), stored_paths, DEVICE)
leaf_probability = leaf_check.leaf_probability(head, features)
is_leaf = readable & (leaf_probability >= LEAF_CUT_OFF)
sharp = sharpness >= BLUR_CUT_OFF if BLUR_ON else np.ones(len(kept), bool)


class Member:
    def __init__(self, path):
        saved = torch.load(path, map_location="cpu", weights_only=False)
        assert saved["classes"] == CLASSES, path
        self.network = training.build_model(len(CLASSES), saved["architecture"], pretrained=False)
        self.network.load_state_dict(saved["state_dict"])
        self.network = self.network.to(DEVICE).eval()
        size = saved["image_size"]
        self.transform = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                                             transforms.Normalize(saved["mean"], saved["std"])])


MODELS = [Member(config.MODELS_DIR / run / "model.pt") for run in MEMBERS]
scores = np.zeros((len(kept), len(CLASSES)), np.float32)
with torch.no_grad():
    for start in tqdm(range(0, len(kept), 32), desc="Ensemble"):
        batch_paths = stored_paths[start:start + 32]
        images = []
        for p in batch_paths:
            with Image.open(p) as im:
                images.append(im.convert("RGB"))
        probs = [torch.softmax(m.network(torch.stack([m.transform(im) for im in images]).to(DEVICE)).float(), dim=1)
                 for m in MODELS]
        scores[start:start + len(images)] = torch.log(torch.stack(probs).mean(dim=0) + 1e-12).cpu().numpy()
np.save(OUT_DIR / "scores.npy", scores)

labels = kept["class"].map(INDEX).to_numpy()
keep_crop = CLASS_CROPS[None, :] == kept["crop"].to_numpy()[:, None]
share = (metrics.softmax(scores / TEMPERATURE) * keep_crop).sum(axis=1)
within = metrics.softmax(np.where(keep_crop, scores, -1e4) / CROP_TEMPERATURE)
confidence, correct = within.max(axis=1), within.argmax(axis=1) == labels
checked_ok = is_leaf & sharp
answered = checked_ok & (share >= 0.5) & (confidence >= THRESHOLD)
kept = kept.assign(leaf_probability=leaf_probability, passed_leaf_check=is_leaf, sharpness=sharpness,
                   passed_blur_check=sharp, confidence=confidence, crop_share=share, correct=correct,
                   answered=answered, predicted=[CLASSES[i] for i in within.argmax(axis=1)])

## Results

*Not checked* = stopped by the leaf check (or the blur check). Among the checked photos:
accuracy with the crop given, the share answered, and the accuracy of the answers. *Wrong per
100* counts wrong answers out of every 100 photos sent, the number a farmer feels.

In [ ]:
def summary(part):
    ok, ans, chk = part["correct"].to_numpy(), part["answered"].to_numpy(), part["passed_leaf_check"].to_numpy()
    chk = chk & part["passed_blur_check"].to_numpy()
    acc_ans = ok[ans].mean() if ans.any() else np.nan
    low, high = (cal.bootstrap(lambda o, a: o[a].mean() if a.any() else np.nan, ok, ans, n=BOOTSTRAP, seed=SEED)
                 if ans.sum() >= 5 else (np.nan, np.nan))
    return {"photos": len(part), "not a leaf": 1 - part["passed_leaf_check"].mean(),
            "too blurred": 1 - part["passed_blur_check"].mean(),
            "accuracy of checked (crop given)": ok[chk].mean() if chk.any() else np.nan,
            "answered": ans.mean(), "accuracy of answered": acc_ans, "answered accuracy low": low,
            "answered accuracy high": high, "wrong per 100": 100 * (ans & ~ok).mean(),
            "to officer or contacts": (chk & ~ans).mean(),
            "ECE (checked)": metrics.expected_calibration_error(part["confidence"].to_numpy()[chk], ok[chk]) if chk.any() else np.nan}


by_class = pd.DataFrame([{"source": s, "class": SHORT[c], **summary(p)} for (s, c), p in kept.groupby(["source", "class"])])
by_class = by_class.set_index(["source", "class"])
overall = pd.DataFrame([{"set": "Bangladesh tomato, 4 classes", **summary(kept[kept["source"] == "tomato_bd_field"])},
                        {"set": "Ghana healthy (tomato and maize)", **summary(kept[kept["source"] == "ghana_healthy"])},
                        {"set": "all", **summary(kept)}]).set_index("set")
display(overall.round(4))
display(by_class.round(4))

# Ghana's leaves are all healthy: any answered disease is a false alarm.
ghana = kept[kept["source"] == "ghana_healthy"]
false_alarms = ghana[ghana["answered"] & ~ghana["correct"]]
print(f"Ghana healthy leaves answered as a disease: {len(false_alarms)} of {len(ghana)}")
display(false_alarms["predicted"].map(SHORT).value_counts().rename("answered as").to_frame())

In [ ]:
# Report only: the same tomato classes on the selection set (notebook 21, deployed rule then
# 0.8 on the old confidence), to see whether tomato is weak beyond the selection set's source.
earlier = pd.read_csv(RESULTS / "health_check" / "a4_selection_by_source_and_class.csv")
display(earlier[earlier["cls"].str.startswith("Tomato")].round(3))
confusion = pd.crosstab(kept["class"].map(SHORT), kept["predicted"].map(SHORT), rownames=["true"], colnames=["predicted (crop given)"])
display(confusion)

## Save the results

In [ ]:
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
overall.to_csv(OUT_DIR / "overall.csv")
by_class.to_csv(OUT_DIR / "by_class.csv")
confusion.to_csv(OUT_DIR / "confusion.csv")
reasons.to_csv(OUT_DIR / "checks.csv", index=False)
sources_table.to_csv(OUT_DIR / "sources.csv", index=False)
final_test.folder_summary(found).to_csv(OUT_DIR / "folders.csv", index=False)
kept.drop(columns=["path", "phash", "phash_variants"]).assign(path=kept["path"].map(lambda p: str(Path(p).relative_to(LOCAL)))
          ).to_csv(OUT_DIR / "photos.csv", index=False)
training.save_json({"scored_once_on": pd.Timestamp.now(tz="UTC").isoformat(), "rerun_reason": RERUN_REASON or None,
                    "pipeline": {"phone": f"{APP_SIDE} px q{APP_QUALITY}", "server_quality": SERVER_QUALITY,
                                 "leaf_cut_off": LEAF_CUT_OFF, "blur_check": BLUR_ON, "blur_cut_off": BLUR_CUT_OFF,
                                 "temperature": TEMPERATURE, "crop_temperature": CROP_TEMPERATURE, "threshold": THRESHOLD,
                                 "model_runs": MEMBERS},
                    "rules_written": "2026-10-31, before any photo was downloaded",
                    "overall": overall.reset_index().to_dict(orient="records")}, OUT_DIR / "decision.json")
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
(REPO_RESULTS / "review_sheets").mkdir(exist_ok=True)
for sheet in sheets:
    shutil.copy(sheet, REPO_RESULTS / "review_sheets" / sheet.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

## Save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
from google.colab import userdata

from src import github_sync

# Drive can hide a folder made just before the remount, so make sure it is there.
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy(ML_DIR / "notebooks" / "21c_final_field_test.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/final-field-test",
    message="Health check Part C: the final field test, scored once",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))